# 03. 의료영상 분할 — MONAI UNet

**분할(segmentation)** 은 영상의 각 픽셀을 장기·병변 등으로 분류하는 작업으로, 의료영상 AI의 핵심입니다.
MONAI의 **UNet** 과 **DiceLoss** 로 분할 모델을 학습합니다.

개념을 명확히 보기 위해 **합성 데이터**(배경 위의 원형 영역)를 만들어 분할합니다. 별도 다운로드가 필요 없습니다.

1. 합성 영상·마스크 생성
2. UNet 모델 구성
3. 학습 (DiceLoss)
4. 평가 (Dice 점수) 및 결과 시각화

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
from tqdm import tqdm

## 1. 합성 영상·마스크 생성

각 영상에 무작위 위치·크기의 원을 그리고, 그 원 영역을 분할 정답(마스크)으로 둡니다.

In [ ]:
def make_sample(size=64, rng=None):
    rng = rng or np.random.default_rng()
    cx, cy = rng.integers(20, size-20, size=2)
    r = rng.integers(8, 16)
    yy, xx = np.mgrid[:size, :size]
    mask = ((xx-cx)**2 + (yy-cy)**2 < r**2).astype('float32')
    image = mask * 1.0 + 0.3 * rng.standard_normal((size, size)).astype('float32')
    return image, mask

rng = np.random.default_rng(0)
N = 120
imgs, masks = zip(*[make_sample(rng=rng) for _ in range(N)])
X = torch.tensor(np.stack(imgs)).unsqueeze(1).float()   # (N,1,64,64)
Y = torch.tensor(np.stack(masks)).unsqueeze(1).float()
n_tr = int(0.8*N)
print('영상:', X.shape, '| 마스크:', Y.shape)

## 2. UNet 모델

MONAI의 UNet은 인코더-디코더 구조로 분할에 특화돼 있습니다. 채널 수·다운샘플 단계를 인자로 정합니다.

In [ ]:
model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64),
    strides=(2, 2),
    num_res_units=2,
).to(device)
print(model.__class__.__name__, '구성 완료')

## 3. 학습

`DiceLoss`는 분할에서 널리 쓰는 손실 함수로, 예측과 정답 영역의 겹침을 최대화합니다.

In [ ]:
loss_fn = DiceLoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Xtr, Ytr = X[:n_tr].to(device), Y[:n_tr].to(device)
pbar = tqdm(range(30), desc='학습')
for epoch in pbar:
    model.train(); optimizer.zero_grad()
    loss = loss_fn(model(Xtr), Ytr)
    loss.backward(); optimizer.step()
    pbar.set_postfix(DiceLoss=f'{loss.item():.4f}')

## 4. 평가 및 시각화

테스트 영상에 대한 Dice 점수(1에 가까울수록 좋음)를 계산하고, 예측 마스크를 정답과 비교합니다.

In [ ]:
model.eval()
Xte, Yte = X[n_tr:].to(device), Y[n_tr:].to(device)
dice = DiceMetric(include_background=True, reduction='mean')
with torch.no_grad():
    pred = (torch.sigmoid(model(Xte)) > 0.5).float()
    dice(y_pred=pred, y=Yte)
print(f'테스트 Dice 점수: {dice.aggregate().item():.3f}')

# 결과 시각화 (첫 샘플)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(Xte[0,0].cpu(), cmap='gray'); axes[0].set_title('입력 영상')
axes[1].imshow(Yte[0,0].cpu(), cmap='gray'); axes[1].set_title('정답 마스크')
axes[2].imshow(pred[0,0].cpu(), cmap='gray'); axes[2].set_title('예측 마스크')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

### 정리

- MONAI UNet과 DiceLoss로 분할 모델을 학습하고 Dice 점수로 평가했습니다.
- 합성 데이터로 분할의 전체 흐름(영상→마스크 예측)을 경험했습니다.
- 실제로는 MedNIST, MSD(Spleen·Liver 등) 같은 데이터셋과 3D UNet으로 확장합니다. (3D는 더 큰 VRAM이 필요합니다.)